In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [4]:
import os

from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [5]:
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

In [6]:
BASE_PATH = "/home/jovyan/work/output/bronze"

In [7]:
SILVER_PATH = "/home/jovyan/work/output/silver"

In [8]:
if not os.path.exists(SILVER_PATH):
    os.makedirs(SILVER_PATH)
    print("Silver folder created")
else:
    print("Silver folder already exists")

Silver folder already exists


In [19]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze
df = spark.read.csv(f"{BASE_PATH}/orders", header=True)

# Remove exact duplicates
df = df.dropDuplicates()

# Type casting
df = df \
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp")) \
    .withColumn("order_approved_at", col("order_approved_at").cast("timestamp")) \
    .withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("timestamp")) \
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
    .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp")) \
    .withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Null handling
df = df.fillna({
    "order_status": "unknown"
})

# Data quality flag
df = df.withColumn(
    "_is_valid",
    when(
        col("order_delivered_customer_date") < col("order_purchase_timestamp"),
        False
    ).otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/orders"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # Standardize schema (VERY IMPORTANT)
    df_old = df_old \
        .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp")) \
        .withColumn("order_approved_at", col("order_approved_at").cast("timestamp")) \
        .withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("timestamp")) \
        .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
        .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp")) \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns (safe practice)
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run (no existing data)

# Dedup (latest record)
window = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write to Silver
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Orders Silver ready (single file)")

Orders Silver ready (single file)


In [12]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze order_items
df = spark.read.csv(f"{BASE_PATH}/order_items", header=True)

# Remove duplicates
df = df.dropDuplicates()

# Type casting
df = df \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("order_item_id", col("order_item_id").cast("int")) \
    .withColumn("shipping_limit_date", col("shipping_limit_date").cast("timestamp")) \
    .withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Join with orders (for validation)
df_orders = spark.read.csv(f"{SILVER_PATH}/orders", header=True)

df = df.join(
    df_orders.select("order_id").withColumnRenamed("order_id", "o_id"),
    df.order_id == col("o_id"),
    "left"
)

# Data quality flag
df = df.withColumn(
    "_is_valid",
    when(col("price") < 0, False)
    .when(col("freight_value") < 0, False)
    .when(col("order_id").isNull(), False)
    .when(col("product_id").isNull(), False)
    .when(col("o_id").isNull(),False)
    .otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/order_items"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # FIX: Standardize schema (same issue as orders)
    df_old = df_old \
        .withColumn("price", col("price").cast("double")) \
        .withColumn("freight_value", col("freight_value").cast("double")) \
        .withColumn("order_item_id", col("order_item_id").cast("int")) \
        .withColumn("shipping_limit_date", col("shipping_limit_date").cast("timestamp")) \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run

# Dedup (latest record per order_id + order_item_id)
window = Window.partitionBy("order_id", "order_item_id") \
               .orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Order Items Silver ready")

Order Items Silver ready


In [14]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze
df = spark.read.csv(f"{BASE_PATH}/customers", header=True)

# Remove exact duplicates
df = df.dropDuplicates()

# Type casting
df = df.withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Data quality flag
df = df.withColumn(
    "_is_valid",
    when(col("customer_id").isNull(), False)
    .when(col("customer_unique_id").isNull(), False)
    .otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/customers"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # FIX: Standardize schema
    df_old = df_old \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run

# Dedup (latest record per customer_id)
window = Window.partitionBy("customer_id") \
               .orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Customers Silver ready")

Customers Silver ready


In [16]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze products
df = spark.read.csv(f"{BASE_PATH}/products", header=True)

# Remove duplicates
df = df.dropDuplicates()

# Type casting
df = df \
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("double")) \
    .withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Null handling
df = df.fillna({
    "product_category_name": "unknown"
})

# Data quality
df = df.withColumn(
    "_is_valid",
    when(col("product_id").isNull(), False)
    .when(col("product_weight_g") < 0, False)
    .otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/products"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # FIX: Standardize schema
    df_old = df_old \
        .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
        .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
        .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
        .withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
        .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
        .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
        .withColumn("product_width_cm", col("product_width_cm").cast("double")) \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run

# Dedup (latest record per product_id)
window = Window.partitionBy("product_id") \
               .orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Products Silver ready")

Products Silver ready


In [17]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze sellers
df = spark.read.csv(f"{BASE_PATH}/sellers", header=True)

# Remove duplicates
df = df.dropDuplicates()

# Type casting
df = df.withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Data quality
df = df.withColumn(
    "_is_valid",
    when(col("seller_id").isNull(), False)
    .otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/sellers"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # FIX: Standardize schema
    df_old = df_old \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run

# Dedup
window = Window.partitionBy("seller_id") \
               .orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Sellers Silver ready")

Sellers Silver ready


In [18]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

# Read Bronze payments
df = spark.read.csv(f"{BASE_PATH}/payments", header=True)

# Remove duplicates
df = df.dropDuplicates()

# Type casting
df = df \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_sequential", col("payment_sequential").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("_ingested_at", col("_ingested_at").cast("timestamp"))

# Null handling
df = df.fillna({
    "payment_type": "unknown"
})

# Data quality
df = df.withColumn(
    "_is_valid",
    when(col("order_id").isNull(), False)
    .when(col("payment_value") < 0, False)
    .otherwise(True)
)

# UPSERT logic
silver_path = f"{SILVER_PATH}/payments"

try:
    df_old = spark.read.csv(silver_path, header=True)

    # FIX: Standardize schema
    df_old = df_old \
        .withColumn("payment_installments", col("payment_installments").cast("int")) \
        .withColumn("payment_sequential", col("payment_sequential").cast("int")) \
        .withColumn("payment_value", col("payment_value").cast("double")) \
        .withColumn("_ingested_at", col("_ingested_at").cast("timestamp")) \
        .withColumn("_is_valid", col("_is_valid").cast("boolean"))

    # Align columns
    df = df.select(sorted(df.columns))
    df_old = df_old.select(sorted(df_old.columns))

    # Union
    df = df.unionByName(df_old, allowMissingColumns=True)

except:
    pass  # First run

# Dedup (composite key)
window = Window.partitionBy("order_id", "payment_sequential") \
               .orderBy(col("_ingested_at").desc())

df_final = df \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Payments Silver ready")

Payments Silver ready
